In [ ]:
%pip install weaviate-client

In [1]:
import weaviate
import json
from weaviate.classes.config import Configure, DataType, Property, VectorDistances
from weaviate.classes.query import Filter, MetadataQuery
from dotenv import load_dotenv
import os


d:\epam\sadhaka\module_3\rag\vectordb\.venv\Lib\site-packages\authlib\integrations\httpx_client\assertion_client.py:5: AuthlibDeprecationWarning: The httpx module is deprecated; please use httpx2 instead.
  from ._compat import httpx2


In [2]:
load_dotenv()

True

In [4]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_REST_ENDPOINT"),
    auth_credentials=os.getenv("WEAVIATE_API_KEY_ADMIN")
)

print("Weaviate ready:", client.is_ready())

Weaviate ready: True


In [ ]:
collection = client.collections.create(
    name="MyCollection",
    properties=[
        Property(name="title", data_type=DataType.TEXT),# will use both the properties to generate vectors
        Property(name="foo", data_type=DataType.INT)
    ],
    vector_config=Configure.Vectors.text2vec_weaviate(
        vector_index_config=Configure.VectorIndex.hfresh(
            distance_metric=VectorDistances.COSINE
        )
    ),
)

print("created collection", collection.name)

created collection MyCollection


In [ ]:
data = [
    {
        "title": "This is ram's boat.",
        "foo": 99,
    },
    {
        "title": "Boat is India's one of the famous headphone brand.",
        "foo": 77
    },
    {
        "title": "Ram is a dump boy. He can't even read a single sentence properly",
        "foo": 55
    },
    {
        "title": "Patrick is my dog name.",
        "foo": 33
    },
    {
        "title": "Dogs are the most loyal animal to humans.",
        "foo": 11        
    },
]

object_uuids = []
for item in data:
    uuid = collection.data.insert(
        properties={
            "title": item["title"],
            "foo": item["foo"],
        }
    )
    object_uuids.append(uuid)

print(f"Inserted {len(object_uuids)} objects.")


Inserted 5 objects.


In [ ]:
print(object_uuids)

[UUID('1c5b33eb-ba7e-4e60-b6af-230c9f9d266f'), UUID('32b141b0-46f8-4c0c-a280-c4df2d9d3035'), UUID('f69ea987-b631-47de-9ff3-48f0cba8a530'), UUID('e0ca83d0-49f2-4a9e-8f4f-abce166c56bb'), UUID('71d07264-0f2b-44f0-b063-29d18f56bf51')]


In [37]:
from weaviate.classes.query import MetadataQuery

#vector similarity search
response = collection.query.near_text(
    query="I need to cross this river.", # search query
    limit=2, # limit
    offset=1, # itne results chood ke dikhao
    include_vector=True, # to get the vector
    return_metadata=MetadataQuery(distance=True), # to include the distance from query
    return_properties=["title"], # specify the properties that you want to get else give all the properties
    filters=Filter.by_property("foo").greater_than(44) #you can apply filter to your query result base on some other property, meta data, time of creation and etc
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)
    print(o.vector["default"][0:5])

{'title': "Ram is a dump boy. He can't even read a single sentence properly"}
0.8668433427810669
[0.04543877765536308, 0.006820629816502333, 0.10435515642166138, -0.029073115438222885, -0.05953250080347061]
{'title': "Boat is India's one of the famous headphone brand."}
0.919644296169281
[-0.008541516028344631, -0.06512215733528137, -0.014314436353743076, 0.05926397442817688, -0.008060021325945854]
